In [ ]:
import os
import tensorflow as tf

os.environ["KERAS_BACKEND"] = "torch"

import torch
import keras
import numpy as np

In [ ]:
import json

with open('./config/keras_nn.json') as keras_nn_config:
    CONFIG = json.load(keras_nn_config)
    print("config loaded")

In [ ]:
SEED = CONFIG["seed"]

keras.utils.set_random_seed(SEED)

In [ ]:
keras.backend.backend()

In [ ]:
%load_ext tensorboard
# now available at http://localhost:6006/?

In [ ]:
from data_loader import get_ml_cup_data
from sklearn.preprocessing import StandardScaler

train_loader, validation_loader, test_loader, input_size, output_size = get_ml_cup_data(CONFIG["batchSize"], scaler=StandardScaler(), validation=True)

In [ ]:
train_loader.dataset.X.shape

In [ ]:
validation_loader.dataset.X.shape

In [ ]:
test_loader.dataset.X.shape

In [ ]:
from keras import Sequential
from keras.layers import Input, Dense

In [ ]:
model = Sequential([
    Input(shape=(input_size,)),
    # Dense layers are fully connected layers
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(output_size)
])

In [ ]:
model.compile(optimizer=CONFIG["optimizer"], loss='mse', metrics=['mae'])

In [ ]:
model.summary()

In [ ]:
from keras.callbacks import EarlyStopping, TensorBoard
import datetime

def log_dir(name, append:str=None):
    BASE = f"logs/{name}/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    if append:
        BASE += "_" + append
    return BASE

In [ ]:
tensorboard_cb = TensorBoard(
    log_dir=log_dir("fit"),
    histogram_freq=1,
    write_graph=True,
    write_images=False
)

early_stopping_cb = EarlyStopping(
    monitor="val_mae",
    patience=10,
    restore_best_weights=True,
)

In [ ]:
tensorboard_cb.log_dir = log_dir("fit", CONFIG["optimizer"])
history = model.fit(train_loader, validation_data=validation_loader, epochs=100, callbacks=[tensorboard_cb, early_stopping_cb])

In [ ]:
%tensorboard --logdir logs/fit

In [ ]:
# evaluate model
results = model.evaluate(test_loader, return_dict=True)
print(results)

In [ ]:
# write logs to a separate folder
writer = tf.summary.create_file_writer(log_dir("eval", CONFIG["optimizer"]))

with writer.as_default():
    for k, v in results.items():
        tf.summary.scalar(k, v, step=0)

writer.close()

In [ ]:
%tensorboard --logdir logs/eval

In [ ]:
## Optuna
import optuna
import tensorflow as tf
from tensorflow import keras

def objective(trial):
    # Suggest hyperparameters
    units1 = trial.suggest_int("units1", 16, 128)
    units2 = trial.suggest_int("units2", 16, 128)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)

    # Build model
    model = Sequential([
        Input(shape=(input_size,)),
        # Dense layers are fully connected layers
        Dense(units1, activation='relu'),
        Dense(units2, activation='relu'),
        Dense(output_size)
    ])

    optimizer = keras.optimizers.SGD(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"]
    )

    tensorboard_cb.log_dir = log_dir("fit", f"OPTUNA_TRIAL#{trial.number}")

    pruning_cb = optuna.integration.KerasPruningCallback(trial, "val_mae")
    
    checkpoint_path = f"checkpoints/optuna_trial_{trial.number}.keras"

    checkpoint_cb = keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_mae",
        mode="min",
        save_best_only=True,
        save_weights_only=False
    )
    
    # Train model
    history = model.fit(
        train_loader,
        validation_data=validation_loader,
        epochs=100,
        verbose=0,
        callbacks=[tensorboard_cb, pruning_cb, checkpoint_cb]
    )

    val_mae = history.history["val_mae"][-1]
    return val_mae

In [ ]:
from optuna.samplers import TPESampler

sampler = TPESampler(seed=SEED)
study = optuna.create_study(sampler=sampler, direction="minimize")
study.optimize(objective, n_trials=30)

print("Best trial:", study.best_trial.params)

In [ ]:
import shutil

def delete_pruned_trial_dirs(root_path, study):
    pruned_numbers = {
        trial.number 
        for trial in study.trials 
        if trial.state == optuna.trial.TrialState.PRUNED
    }

    removed = []

    if not os.path.isdir(root_path):
        raise ValueError(f"{root_path} is not a valid directory.")

    for entry in os.listdir(root_path):
        full_path = os.path.join(root_path, entry)

        if entry.startswith("."):
            continue  # skip hidden folders
            
        if entry.count("_OPTUNA_TRIAL#") == 1:
            try:
                trial_num = int(entry.split("_OPTUNA_TRIAL#")[1])
            except ValueError:
                continue

            if trial_num in pruned_numbers:
                shutil.rmtree(full_path)
                removed.append(entry)

    return removed

In [ ]:
delete_pruned_trial_dirs(root_path="logs/fit", study=study)

In [ ]:
study.best_trial

In [ ]:
best_trial = study.best_trial
best_model_path = f"checkpoints/optuna_trial_{best_trial.number}.keras"
best_model = keras.models.load_model(best_model_path)

In [ ]:
test_loss, test_mae = best_model.evaluate(test_loader)

In [ ]:
writer = tf.summary.create_file_writer(log_dir("eval", f"OPTUNA_TRIAL#{best_trial.number}"))
with writer.as_default():
    tf.summary.scalar("loss", test_loss, step=0)
    tf.summary.scalar("mae",  test_mae,  step=0)
    writer.flush()

In [ ]:
%tensorboard --logdir logs/eval